In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================

# Force reloading of environment variables to ensure local runtime context shifts
# or secrets rotation are immediately captured without restarting the process.
load_dotenv(override=True)

# ==============================================================================
# 2. DATABASE CONNECTION SETUP
# ==============================================================================

# Construct standard connection URI string for PostgreSQL dialect.
# SQLAlchemy engine decouples connection management and provisions a thread-safe connection pool.
DATABASE_URL = f"postgresql://{os.getenv('user')}:{os.getenv('password')}@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}"
engine = create_engine(DATABASE_URL)

# ==============================================================================
# 3. SCHEMA DEFINITION
# ==============================================================================

# Strict sequential field arrangement mapped identically to the destination DDL schema.
# This array acts as the source of truth for programmatic structural validation.
cols_sql = [
    'ticker', 'asset_name', 'sector', 'exchange', 'currency', 'beta',
    'dividend_yield', 'trailingpe', 'pricetobook', 'fcf_yield',
    'revenuegrowth', 'earningsgrowth', 'forwardeps', 'payoutratio',
    'asset_type', 'value_score', 'growth_score', 'dividend_score'  
]

# ==============================================================================
# 4. DATA ACQUISITION
# ==============================================================================

# Ingest raw extracted metrics payload.
# Enforce case normalization (lowercase) on source headers to prevent string-matching
# friction against lowercase-native PostgreSQL identifiers.
df = pd.read_csv('../1_data_extraction/data/assets_info.csv')
df.columns = df.columns.str.lower()

# ==============================================================================
# 5. DATA ALIGNMENT & REINDEXING
# ==============================================================================

# Programmatically align DataFrame memory layout with the core SQL columns constraint list.
# Features missing from the source pipeline are padded with NaN values to prevent shape mismatches,
# while extraneous source features are safely pruned to protect structural integrity.
df = df.reindex(columns=cols_sql)

# ==============================================================================
# 6. INTEGRITY CLEANING
# ==============================================================================

# Enforce relational data constraints before initiating network transmission.
# Drops unidentifiable financial entries where 'asset_name' is missing, bypassing
# 'NotNullViolation' rollbacks at the database server level.
df = df.dropna(subset=['asset_name'])

# ==============================================================================
# 7. DATA LOADING (ETL FINAL STAGE)
# ==============================================================================

try:
    # Bulk load finalized structured elements into the relational data store.
    # 'method=multi' batches insertions inside parameterized execution plans,
    # and 'chunksize=500' caps memory consumption per network roundtrip for high stability.
    df.to_sql(
        name='assets', 
        con=engine, 
        if_exists='append', 
        index=False, 
        method='multi', 
        chunksize=500
    )
    print(f"Success! {len(df)} assets have been successfully loaded.")

except Exception as e:
    # Gracefully intercept database transaction faults or connection timeouts,
    # exposing raw error traces for diagnostic transparency without crashing the lifecycle.
    print(f"Database Integrity Error: {e}")

Success! 49 assets have been successfully loaded.
